# Naive Bias Model:-

- BigQuery ML does not have a native model_type = 'NAIVE_BAYES'.

- In machine learning, Naive Bayes is traditionally used for NLP / Text Classification (like Spam vs. Ham) by converting text into word frequencies (TF-IDF or Bag-of-Words). In BigQuery ML, the official and standard way to build a spam text classifier is using LOGISTIC_REG combined with built-in TF-IDF text tokenization (ML.TF_IDF + ML.NGRAMS) in the TRANSFORM clause.

- Below is the complete BigQuery pipeline:
    1. **Method 1 (Recommended)**: The production-grade BQML Text Classifier using TF-IDF + Logistic Regression.
    2. **Method 2 (Bonus)**: Pure SQL Implementation of the exact Bayes' Theorem mathematical formula.

# Step by Step Complete Guide of Naive Bias Model:-

- **Step 1**: Create Schema / Dataset

```sql
-- Create a schema/dataset in the US region
CREATE SCHEMA IF NOT EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox`
OPTIONS(
  location = 'US'
);
```

> gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_iris

---

- **Step 2**: Cleanse and Preprocess Data -> Clean up column names, strip extra spaces, and drop any empty unnamed trailing columns.

```sql
-- Cleanse data: normalize column names to 'label' and 'message'
CREATE OR REPLACE TABLE `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_spam` AS
SELECT
  -- Normalize label ('ham' = normal message, 'spam' = unwanted message)
  LOWER(TRIM(CAST(v1 AS STRING))) AS label,
  
  -- The SMS text message content
  TRIM(CAST(v2 AS STRING)) AS message
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_spam`
WHERE
  v1 IS NOT NULL AND v2 IS NOT NULL;
```

---


- **Step 3**: Train the Text Classifier (TF-IDF + BQML) -> 
In this query, the TRANSFORM block extracts TF-IDF unigrams and bigrams (word frequencies) directly from the message text before passing them into the classifier.

```sql
-- Train the Text Classifier with TF-IDF Vectorization
CREATE OR REPLACE MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.sms_spam_model`
TRANSFORM(
  label,
  -- Step A: Tokenize text into words & calculate TF-IDF weights (Bag of Words)
  ML.TF_IDF(
    ML.NGRAMS(ML.STANDARD_TOKENIZE(LOWER(message)), [1, 2]),
    top_k => 2000,
    frequency_threshold => 2
  ) OVER() AS text_tfidf
)
OPTIONS (
  model_type = 'LOGISTIC_REG',
  input_label_cols = ['label'],
  
  -- Imbalance handling (ham messages outnumber spam ~6:1)
  auto_class_weights = TRUE,
  
  -- 80% train / 20% test split
  data_split_method = 'RANDOM',
  data_split_eval_fraction = 0.2,
  
  -- L2 Regularization to prevent overfitting on rare words
  l2_reg = 0.5
) AS
SELECT
  label,
  message
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_spam`;
```

---


- **Step 4**: Evaluate Model Performance -> Check Accuracy, Precision, Recall, and ROC-AUC on the test dataset.

```sql
-- Evaluate classification performance
SELECT
  *
FROM
  ML.EVALUATE(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.sms_spam_model`);
```

---


- **Step 5**: Confusion Matrix -> In Spam Detection, we especially want to minimize False Positives (important emails/SMS from friends falsely flagged as spam).

```sql
-- Check False Positives vs False Negatives
SELECT
  *
FROM
  ML.CONFUSION_MATRIX(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.sms_spam_model`);
```

---


- **Step 6**: Make Predictions on New SMS Messages -> Test the model with a clear spam message and a clear personal (ham) message:

```sql
-- Test predictions on unseen text messages
SELECT
  predicted_label,
  predicted_label_probs,
  message
FROM
  ML.PREDICT(
    MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.sms_spam_model`,
    (
      -- Message 1: Obvious Spam message
      SELECT 
        CAST('URGENT! You have won a 1 week FREE vacation to Hawaii. Call 0800-123456 now to claim prize!' AS STRING) AS message
      UNION ALL
      -- Message 2: Normal Ham message
      SELECT 
        CAST('Hey mate, are we still meeting up for coffee after work today?' AS STRING) AS message
      UNION ALL
      -- Message 3: Bank verification (subtle)
      SELECT 
        CAST('Your security code is 492018. Do not share this code with anyone.' AS STRING) AS message
    )
  );
```

---


- **Step 7**: Bonus — Pure SQL Naive Bayes Formula

```sql
-- Word probability analysis (Pure Naive Bayes components)
WITH words_unpacked AS (
  SELECT
    label,
    word
  FROM
    `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_spam`,
    UNNEST(ML.STANDARD_TOKENIZE(LOWER(message))) AS word
)
SELECT
  word,
  COUNTIF(label = 'spam') AS spam_count,
  COUNTIF(label = 'ham') AS ham_count,
  -- Likelihood of word given it is Spam: P(Word | Spam)
  ROUND(COUNTIF(label = 'spam') / (SELECT COUNT(*) FROM words_unpacked WHERE label = 'spam'), 4) AS p_word_given_spam,
  -- Likelihood of word given it is Ham: P(Word | Ham)
  ROUND(COUNTIF(label = 'ham') / (SELECT COUNT(*) FROM words_unpacked WHERE label = 'ham'), 4) AS p_word_given_ham
FROM
  words_unpacked
GROUP BY
  word
HAVING
  spam_count + ham_count >= 10
ORDER BY
  spam_count DESC
LIMIT 15;
```

---

- **Step 8**: Clean Up All Resources (Optional)

```sql
-- Cleanup resources
DROP MODEL IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.sms_spam_model`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_spam`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_spam`;
DROP SCHEMA IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox` CASCADE;
```

---